# INST326 — Week 8 Exercises: Inheritance & Polymorphism (Library Management)

**Focus (Week 8 only):** subclassing, method overriding, `super()`, polymorphism via common method names, and composition vs. inheritance decisions in a small codebase.

**Out of scope (Week 9+):** abstract base classes, interfaces/protocols, multiple inheritance/mixins, advanced design patterns, decorators beyond basics, context managers beyond prior weeks, dependency injection, property descriptors beyond simple use.

> Context: Use the Library Management domain—books, members, loans, fines—to complete the tasks. Stick to basic single inheritance and straightforward overrides.


### Starter Scaffold (Week‑8‑safe)

Below is minimal starter code from prior weeks, extended slightly for Week 8. Feel free to modify it for the exercises. Avoid Week 9+ topics.


In [23]:
from __future__ import annotations
from dataclasses import dataclass
from datetime import datetime, timedelta
from typing import Dict, Optional, List

# --- Exceptions from Week 7 (basic) ---
class LibraryError(Exception): ...
class DuplicateBookError(LibraryError): ...
class OverdueLoanError(LibraryError): ...

# --- Base domain classes (no ABCs) ---
@dataclass
class Book:
    isbn: str
    title: str
    copies: int = 1

    def loan_period_days(self) -> int:
        """Default loan period for a generic book."""
        return 14

    def describe(self) -> str:
        return f"Book<{self.isbn}>: {self.title} (copies={self.copies})"

@dataclass
class Member:
    member_id: str
    email: str

    def max_concurrent_loans(self) -> int:
        return 5

    def describe(self) -> str:
        return f"Member<{self.member_id}>"

@dataclass
class Loan:
    isbn: str
    member_id: str
    due_date: datetime
    returned: bool = False

    def mark_returned(self) -> None:
        self.returned = True

class Catalog:
    def __init__(self):
        self._books: Dict[str, Book] = {}

    def add_book(self, book: Book) -> None:
        if book.isbn in self._books:
            raise DuplicateBookError(f"ISBN already exists: {book.isbn}")
        if book.copies < 0:
            raise ValueError("copies must be non-negative")
        self._books[book.isbn] = book

    def get_book(self, isbn: str) -> Optional[Book]:
        return self._books.get(isbn)

class LoanDesk:
    """Very small service; deliberately simple for Week 8 examples."""
    def __init__(self, catalog: Catalog):
        self.catalog = catalog
        self.loans: List[Loan] = []

    def checkout(self, member: Member, book: Book) -> Loan:
        # naive stock check
        if book.copies <= 0:
            raise LibraryError("no available copies")
        book.copies -= 1
        due = datetime.now() + timedelta(days=book.loan_period_days())
        loan = Loan(isbn=book.isbn, member_id=member.member_id, due_date=due)
        self.loans.append(loan)
        return loan

    def checkin(self, loan: Loan) -> None:
        if not loan.returned:
            b = self.catalog.get_book(loan.isbn)
            if b:
                b.copies += 1
            loan.mark_returned()


## 1) Subclass a Book type

Create a subclass `PrintedBook(Book)` that overrides `loan_period_days()` to 21 days and `describe()` to include the word 'Printed'.

In [ ]:
# Your code here
class PrintedBook(Book):
    def loan_period_days(self) -> int:
        return 21
    
    def describe(self) -> str:
        return f"PrintedBook<{self.isbn}>: {self.title} (copies={self.copies})"

pb = PrintedBook("111", "Intro to Python", 2)
print(f"Loan period: {pb.loan_period_days()} days")  
print(f"Description: {pb.describe()}")               

Loan period: 21 days
Description: PrintedBook<111>: Intro to Python (copies=2)


## 2) Another Book subtype

Create `EBook(Book)` that has an extra attribute `file_size_mb: float` (add to `__init__`), uses a 14‑day loan, and overrides `describe()` to show the size.

In [ ]:
# Your code here
class EBook(Book):
    def __init__(self, isbn: str, title: str, copies: int = 1, file_size_mb: float = 0.0):
        super().__init__(isbn, title, copies)
        self.file_size_mb = file_size_mb
    
    def loan_period_days(self) -> int:
        return 14
    
    def describe(self) -> str:
        return f"EBook<{self.isbn}>: {self.title} (copies={self.copies}, size={self.file_size_mb}MB)"

# Example:
eb = EBook("222", "Digital Python Guide", 5, 15.5)
print(f"Loan period: {eb.loan_period_days()} days")
print(f"Description: {eb.describe()}")

Loan period: 14 days
Description: EBook<222>: Digital Python Guide (copies=5, size=15.5MB)


## 3) Override with super()

Create `AudioBook(Book)` with extra field `duration_min: int`. Override `describe()` to start with `super().describe()` and append `duration_min`.

In [ ]:
# Your code here
class AudioBook(Book):
    def __init__(self, isbn: str, title: str, copies: int = 1, duration_min: int = 0):
        super().__init__(isbn, title, copies)
        self.duration_min = duration_min
    
    def describe(self) -> str:
        base_desc = super().describe()
        return f"{base_desc} (duration={self.duration_min}min)"

# Example:
ab = AudioBook("333", "Python Audio Tutorial", 3, 240)
print(f"Description: {ab.describe()}")

Description: Book<333>: Python Audio Tutorial (copies=3) (duration=240min)


## 4) Non‑circulating subclass

Create `ReferenceBook(Book)` that **cannot** be checked out. Override `loan_period_days()` to return `0`. In `LoanDesk.checkout`, demonstrate a guard that raises `LibraryError('non-circulating')` if the period is 0.

In [ ]:
# Your code here
class ReferenceBook(Book):
    def loan_period_days(self) -> int:
        return 0

# Update LoanDesk.checkout to guard non-circulating books
class ExtendedLoanDesk(LoanDesk):
    def checkout(self, member: Member, book: Book) -> Loan:
        # Check if book is non-circulating
        if book.loan_period_days() == 0:
            raise LibraryError('non-circulating')
        
        # Use original checkout logic
        return super().checkout(member, book)

# Example:
ref_book = ReferenceBook("444", "Encyclopedia", 1)
print(f"Reference book loan period: {ref_book.loan_period_days()} days")

# Test the guard
catalog = Catalog()
catalog.add_book(ref_book)
desk = ExtendedLoanDesk(catalog)
member = Member("M001", "test@example.com")

try:
    desk.checkout(member, ref_book)
except LibraryError as e:
    print(f"Expected error: {e}")

Reference book loan period: 0 days
Expected error: non-circulating


## 5) Member specialization

Create `Student(Member)` and `Staff(Member)`. Students can have 5 concurrent loans; staff 10. Override `max_concurrent_loans()` accordingly.

In [28]:
# Your code here
class Student(Member):
    def max_concurrent_loans(self) -> int:
        return 5

class Staff(Member):
    def max_concurrent_loans(self) -> int:
        return 10

# Example:
student = Student("S001", "student@university.edu")
staff = Staff("F001", "faculty@university.edu")

print(f"Student max loans: {student.max_concurrent_loans()}")
print(f"Staff max loans: {staff.max_concurrent_loans()}")

Student max loans: 5
Staff max loans: 10


## 6) Polymorphic fine calculation

Write a function `late_fee(book: Book, days_late: int) -> float` that uses polymorphic behavior:
- PrintedBook: $0.25/day
- EBook: $0.10/day
- AudioBook: $0.15/day
- Fallback (Book): $0.20/day
Use `isinstance` checks only; do not modify the classes for this one.

In [ ]:
# Your code here
def late_fee(book: Book, days_late: int) -> float:
    if isinstance(book, PrintedBook):
        return 0.25 * days_late
    elif isinstance(book, EBook):
        return 0.10 * days_late
    elif isinstance(book, AudioBook):
        return 0.15 * days_late
    else:
        return 0.20 * days_late

# Example:
pb = PrintedBook("111", "Intro to Python", 2)
eb = EBook("222", "Digital Guide", 5, 15.5)

days_late = 5
print(f"PrintedBook late fee (5 days): ${late_fee(pb, days_late):.2f}")
print(f"EBook late fee (5 days): ${late_fee(eb, days_late):.2f}")

PrintedBook late fee (5 days): $1.25
EBook late fee (5 days): $0.50


## 7) Polymorphism without isinstance

Refactor your approach so **each subclass** implements `daily_late_fee()` and `late_fee(days_late)` (calling `daily_late_fee()`), then write a single function `compute_fee(book: Book, days_late: int)` that calls `book.late_fee(days_late)` without type checks.

In [31]:

Book.daily_late_fee = lambda self: 0.20
Book.late_fee = lambda self, days_late: self.daily_late_fee() * days_late


PrintedBook.daily_late_fee = lambda self: 0.25


EBook.daily_late_fee = lambda self: 0.10


AudioBook.daily_late_fee = lambda self: 0.15

def compute_fee(book: Book, days_late: int) -> float:

    return book.late_fee(days_late)

# Example:
pb = PrintedBook("111", "Intro to Python", 2)
eb = EBook("222", "Digital Guide", 5, 15.5)

days_late = 5
print(f"PrintedBook late fee (5 days): ${compute_fee(pb, days_late):.2f}")
print(f"EBook late fee (5 days): ${compute_fee(eb, days_late):.2f}")


PrintedBook late fee (5 days): $1.25
EBook late fee (5 days): $0.50


## 8) Overriding __str__

Override `__str__` in `Book` to return `<Book isbn=... title=...>`. Override it in one subclass to include subtype info, e.g., `<PrintedBook isbn=...>`.

In [32]:

Book.__str__ = lambda self: f"<Book isbn={self.isbn} title={self.title}>"


PrintedBook.__str__ = lambda self: f"<PrintedBook isbn={self.isbn} title={self.title}>"

regular_book = Book("444", "Regular Book", 1)
pb = PrintedBook("111", "Intro to Python", 2)

print(f"Regular book: {regular_book}")
print(f"Printed book: {pb}")

Regular book: <Book isbn=444 title=Regular Book>
Printed book: <PrintedBook isbn=111 title=Intro to Python>


## 9) Composition vs. inheritance

Create a small `Notifier` class with method `notify(member: Member, message: str)`. Demonstrate **composition** by adding a `notifier` attribute to `LoanDesk` and using it during checkout to acknowledge a loan. Keep `Notifier` very simple (e.g., print or collect messages).

In [ ]:
# Your code here
class Notifier:
    def __init__(self):
        self.messages = []
    
    def notify(self, member: Member, message: str) -> None:
        notification = f"To {member.email}: {message}"
        print(notification)
        self.messages.append(notification)


class NotifyingLoanDesk(LoanDesk):
    def __init__(self, catalog: Catalog, notifier: Optional[Notifier] = None):
        super().__init__(catalog)
        self.notifier = notifier or Notifier()
    
    def checkout(self, member: Member, book: Book) -> Loan:

        loan = super().checkout(member, book)
        
        self.notifier.notify(member, f"You checked out '{book.title}' due on {loan.due_date.strftime('%Y-%m-%d')}")
        
        return loan

# Example:
catalog = Catalog()
book = PrintedBook("555", "Python Patterns", 3)
catalog.add_book(book)

notifier = Notifier()
desk = NotifyingLoanDesk(catalog, notifier)
member = Student("S001", "student@university.edu")

loan = desk.checkout(member, book)
print(f"Total notifications sent: {len(notifier.messages)}")

To student@university.edu: You checked out 'Python Patterns' due on 2025-11-30
Total notifications sent: 1


## 10) Enforcing limits polymorphically

Modify `LoanDesk.checkout` to check a member's current active loans (for that member_id) and compare to `member.max_concurrent_loans()` before allowing checkout. Demonstrate with a `Student` hitting the limit and a `Staff` not hitting it.

In [33]:
# Your code here

class LimitEnforcingLoanDesk(LoanDesk):
    def checkout(self, member: Member, book: Book) -> Loan:
        active_loans = sum(1 for loan in self.loans 
                          if loan.member_id == member.member_id and not loan.returned)
        
        if active_loans >= member.max_concurrent_loans():
            raise LibraryError(f"Member {member.member_id} has reached loan limit of {member.max_concurrent_loans()}")
        
        return super().checkout(member, book)

catalog = Catalog()
for i in range(10):
    book = Book(f"ISBN-{i}", f"Book {i}", 1)
    catalog.add_book(book)

desk = LimitEnforcingLoanDesk(catalog)
student = Student("S001", "student@university.edu")  
staff = Staff("F001", "faculty@university.edu")     

try:
    for i in range(6):  
        book = catalog.get_book(f"ISBN-{i}")
        loan = desk.checkout(student, book)
        print(f"Student checked out book {i}")
except LibraryError as e:
    print(f"Student limit reached: {e}")


try:
    for i in range(6, 8):  
        book = catalog.get_book(f"ISBN-{i}")
        loan = desk.checkout(staff, book)
        print(f"Staff checked out book {i}")
except LibraryError as e:
    print(f"Staff error: {e}")

Student checked out book 0
Student checked out book 1
Student checked out book 2
Student checked out book 3
Student checked out book 4
Student limit reached: Member S001 has reached loan limit of 5
Staff checked out book 6
Staff checked out book 7


## 11) Subclass‑specific behavior

Add a method `download_link()` to `EBook` returning a fake URL string using the ISBN. Do not add this to `Book` or other subclasses. Show a short snippet where you use duck typing safely by checking `hasattr` before calling.

In [34]:
# Your code here

EBook.download_link = lambda self: f"https://library.example.com/downloads/{self.isbn}.pdf"

# Duck typing demo
def get_download_info(book: Book) -> str:
    if hasattr(book, 'download_link'):
        return f"Download available: {book.download_link()}"
    else:
        return "No download available for this book type"

# Example:
eb = EBook("222", "Digital Guide", 5, 15.5)
pb = PrintedBook("111", "Intro to Python", 2)
ab = AudioBook("333", "Audio Tutorial", 3, 240)

print(f"EBook: {get_download_info(eb)}")
print(f"PrintedBook: {get_download_info(pb)}")
print(f"AudioBook: {get_download_info(ab)}")

EBook: Download available: https://library.example.com/downloads/222.pdf
PrintedBook: No download available for this book type
AudioBook: No download available for this book type


## 12) Polymorphic loan period by member type

Some libraries extend loan periods for `Staff`. Implement `effective_loan_period(book: Book, member: Member) -> int`:
- start from `book.loan_period_days()`
- if `isinstance(member, Staff)`, add +7 days
Return the resulting days.

In [ ]:
# Your code here
def effective_loan_period(book: Book, member: Member) -> int:
    base_period = book.loan_period_days()
    
    # Staff get an extra 7 days
    if isinstance(member, Staff):
        return base_period + 7
    
    return base_period

# Example:
pb = PrintedBook("111", "Intro to Python", 2)  # 21 days
eb = EBook("222", "Digital Guide", 5, 15.5)    # 14 days

student = Student("S001", "student@university.edu")
staff = Staff("F001", "faculty@university.edu")

print(f"PrintedBook for Student: {effective_loan_period(pb, student)} days")
print(f"PrintedBook for Staff: {effective_loan_period(pb, staff)} days")
print(f"EBook for Student: {effective_loan_period(eb, student)} days")
print(f"EBook for Staff: {effective_loan_period(eb, staff)} days")

PrintedBook for Student: 21 days
PrintedBook for Staff: 28 days
EBook for Student: 14 days
EBook for Staff: 21 days


## 13) Override equality semantics (dataclass)

For `Book`, override `__eq__` so that books are considered equal iff ISBNs match (ignore title/copies). Write quick tests comparing a `PrintedBook` and `EBook` with the same ISBN—they should be equal by ISBN.

In [ ]:
# Your code here
Book.__eq__ = lambda self, other: isinstance(other, Book) and self.isbn == other.isbn
Book.__hash__ = lambda self: hash(self.isbn)


same_isbn = "123-456-789"
pb = PrintedBook(same_isbn, "Python Guide", 2)
eb = EBook(same_isbn, "Python Guide Digital", 5, 15.5)
different_book = PrintedBook("999-888-777", "Other Book", 1)

print(f"PrintedBook == EBook (same ISBN): {pb == eb}")       
print(f"PrintedBook == PrintedBook (different ISBN): {pb == different_book}")
print(f"PrintedBook ISBN: {pb.isbn}, EBook ISBN: {eb.isbn}")
print(f"Books with same ISBN are considered equal regardless of type or other attributes")

PrintedBook == EBook (same ISBN): True
PrintedBook == PrintedBook (different ISBN): False
PrintedBook ISBN: 123-456-789, EBook ISBN: 123-456-789
Books with same ISBN are considered equal regardless of type or other attributes


## 14) Draft a small class hierarchy diagram (markdown)

In **markdown**, sketch a tiny hierarchy diagram for `Book <- PrintedBook | EBook | AudioBook | ReferenceBook` and `Member <- Student | Staff`. No code—just a clear diagram using text/ASCII.

In [ ]:
# Class Hierarchy Diagram

```
Book (base class)
├── PrintedBook
├── EBook
├── AudioBook
└── ReferenceBook

Member (base class)
├── Student
└── Staff
```

**Inheritance Relationships:**
- All book types inherit from `Book` and override methods like `loan_period_days()` and `describe()`
- `Student` and `Staff` inherit from `Member` and override `max_concurrent_loans()`
- Each subclass specializes behavior while maintaining the base interface

## 15) Replace conditional with polymorphism

Currently, `LoanDesk.checkout` always uses `book.loan_period_days()`. Add an overridable method `checkout_message()` to `Book` and override it in at least two subclasses to customize the user‑facing message returned by `LoanDesk.checkout` (e.g., 'Enjoy your audiobook!'). Show the different messages without `if/elif` chains.

In [ ]:
# Your code here

Book.checkout_message = lambda self: f"You've checked out '{self.title}'. Enjoy reading!"


PrintedBook.checkout_message = lambda self: f"You've checked out the printed book '{self.title}'. Happy reading!"
EBook.checkout_message = lambda self: f"Your digital book '{self.title}' is ready for download. Enjoy!"
AudioBook.checkout_message = lambda self: f"Your audiobook '{self.title}' is ready to listen. Enjoy the experience!"
ReferenceBook.checkout_message = lambda self: f"Reference book '{self.title}' - please use in library only."

class MessageLoanDesk(LoanDesk):
    def checkout(self, member: Member, book: Book) -> tuple[Loan, str]:
        """Return both the loan and the customized message."""
        loan = super().checkout(member, book)
        message = book.checkout_message() 
        return loan, message


catalog = Catalog()
books = [
    PrintedBook("111", "Python Guide", 2),
    EBook("222", "Digital Python", 5, 15.5),
    AudioBook("333", "Python Audio", 3, 240),
    Book("444", "Regular Book", 1)
]

for book in books:
    catalog.add_book(book)

desk = MessageLoanDesk(catalog)
member = Student("S001", "student@university.edu")

for book in books:
    try:
        loan, message = desk.checkout(member, book)
        print(f"{book.__class__.__name__}: {message}")
    except LibraryError as e:
        print(f"Error with {book.__class__.__name__}: {e}")

PrintedBook: You've checked out the printed book 'Python Guide'. Happy reading!
EBook: Your digital book 'Digital Python' is ready for download. Enjoy!
AudioBook: Your audiobook 'Python Audio' is ready to listen. Enjoy the experience!
Book: You've checked out 'Regular Book'. Enjoy reading!


## 16) Subclass‑specific stock policy

Override `LoanDesk.checkout` to deny checkout of `EBook` if `copies < 0` (simulate licensing depletion), but allow `PrintedBook` as long as `copies > 0`. Implement this by relying on each subclass's own `can_checkout(stock: int) -> bool` method. Default in `Book` should be `stock > 0`.

In [35]:
# Your code here

Book.can_checkout = lambda self, stock: stock > 0


EBook.can_checkout = lambda self, stock: stock >= 0

class StockPolicyLoanDesk(LoanDesk):
    def checkout(self, member: Member, book: Book) -> Loan:
        """Use polymorphic stock policy check."""
        if not book.can_checkout(book.copies):
            raise LibraryError(f"Cannot checkout {book.title}: stock policy violation")
        
        return super().checkout(member, book)

catalog = Catalog()


pb_good = PrintedBook("111", "Printed Book Good", 1) 
pb_bad = PrintedBook("112", "Printed Book Bad", 0)    
eb_good = EBook("222", "EBook Good", 1, 15.5)        
eb_zero = EBook("223", "EBook Zero", 0, 15.5)        

for book in [pb_good, pb_bad, eb_good, eb_zero]:
    catalog.add_book(book)

desk = StockPolicyLoanDesk(catalog)
member = Student("S001", "student@university.edu")

for book in [pb_good, pb_bad, eb_good, eb_zero]:
    try:
        loan = desk.checkout(member, book)
        print(f"Successfully checked out {book.__class__.__name__}: {book.title} (stock={book.copies})")
    except LibraryError as e:
        print(f" Failed to checkout {book.__class__.__name__}: {e}")

Successfully checked out PrintedBook: Printed Book Good (stock=0)
 Failed to checkout PrintedBook: Cannot checkout Printed Book Bad: stock policy violation
Successfully checked out EBook: EBook Good (stock=0)
 Failed to checkout EBook: no available copies


## 17) Sorting polymorphically

Create a list mixing `PrintedBook`, `EBook`, and `AudioBook`. Implement a function `sort_books_for_display(books: list[Book]) -> list[Book]` that sorts by this precedence: Printed first, then EBook, then AudioBook; ties broken by title. Use a key function that relies on `isinstance` or a small polymorphic `display_rank()` method.

In [36]:
# Your code here
def sort_books_for_display(books: list[Book]) -> list[Book]:

    
    def get_sort_key(book: Book) -> tuple[int, str]:
     
        if isinstance(book, PrintedBook):
            rank = 1
        elif isinstance(book, EBook):
            rank = 2
        elif isinstance(book, AudioBook):
            rank = 3
        else:  
            rank = 4
        
        return (rank, book.title)
    
    return sorted(books, key=get_sort_key)


Book.display_rank = lambda self: 4  
PrintedBook.display_rank = lambda self: 1
EBook.display_rank = lambda self: 2
AudioBook.display_rank = lambda self: 3

def sort_books_polymorphic(books: list[Book]) -> list[Book]:
    return sorted(books, key=lambda book: (book.display_rank(), book.title))

# Example:
mixed_books = [
    AudioBook("333", "C Audio Guide", 3, 240),
    PrintedBook("111", "A Printed Guide", 2),
    EBook("222", "B Digital Guide", 5, 15.5),
    PrintedBook("444", "Z Printed Advanced", 1),
    AudioBook("555", "A Audio Basics", 2, 180),
    Book("666", "M Regular Book", 1)
]

print("Original order:")
for book in mixed_books:
    print(f"  {book.__class__.__name__}: {book.title}")

print("\nSorted with isinstance:")
sorted_books = sort_books_for_display(mixed_books)
for book in sorted_books:
    print(f"  {book.__class__.__name__}: {book.title}")

print("\nSorted with polymorphic method:")
sorted_poly = sort_books_polymorphic(mixed_books)
for book in sorted_poly:
    print(f"  {book.__class__.__name__}: {book.title}")

Original order:
  AudioBook: C Audio Guide
  PrintedBook: A Printed Guide
  EBook: B Digital Guide
  PrintedBook: Z Printed Advanced
  AudioBook: A Audio Basics
  Book: M Regular Book

Sorted with isinstance:
  PrintedBook: A Printed Guide
  PrintedBook: Z Printed Advanced
  EBook: B Digital Guide
  AudioBook: A Audio Basics
  AudioBook: C Audio Guide
  Book: M Regular Book

Sorted with polymorphic method:
  PrintedBook: A Printed Guide
  PrintedBook: Z Printed Advanced
  EBook: B Digital Guide
  AudioBook: A Audio Basics
  AudioBook: C Audio Guide
  Book: M Regular Book


## 18) Minimal polymorphic report

Write `summarize_books(books: list[Book]) -> list[str]` that returns `describe()` for each. Show that the correct overridden `describe()` is used without `if/elif`.

In [37]:
# Your code here
def summarize_books(books: list[Book]) -> list[str]:
    return [book.describe() for book in books]

# Example:
sample_books = [
    PrintedBook("111", "Printed Python Guide", 2),
    EBook("222", "Digital Programming", 5, 20.5),
    AudioBook("333", "Audio Tutorial", 3, 300),
    Book("444", "Regular Book", 1)
]

print("Polymorphic book summaries:")
summaries = summarize_books(sample_books)
for i, summary in enumerate(summaries, 1):
    print(f"{i}. {summary}")

print("\nNote: Each book type's overridden describe() method is called automatically!")

Polymorphic book summaries:
1. PrintedBook<111>: Printed Python Guide (copies=2)
2. EBook<222>: Digital Programming (copies=5, size=20.5MB)
3. Book<333>: Audio Tutorial (copies=3) (duration=300min)
4. Book<444>: Regular Book (copies=1)

Note: Each book type's overridden describe() method is called automatically!


## 19) Unit test: overriding works

Using `unittest`, add a small test class that checks `loan_period_days()` for `PrintedBook` (21) and `ReferenceBook` (0), and that `__str__` includes the subclass name for one subtype.

In [38]:
# Your code here
import unittest

class TestWeek8Inheritance(unittest.TestCase):
    def test_loan_periods_and_str(self):
        
        pb = PrintedBook("111", "Printed Book", 2)
        rb = ReferenceBook("222", "Reference Book", 1)
        regular_book = Book("333", "Regular Book", 1)
        
        self.assertEqual(pb.loan_period_days(), 21, "PrintedBook should have 21-day loan period")
        self.assertEqual(rb.loan_period_days(), 0, "ReferenceBook should have 0-day loan period")
        self.assertEqual(regular_book.loan_period_days(), 14, "Regular Book should have 14-day loan period")
        

        pb_str = str(pb)
        self.assertIn("PrintedBook", pb_str, "__str__ should include subclass name")
        self.assertIn("111", pb_str, "__str__ should include ISBN")
        
        regular_str = str(regular_book)
        self.assertIn("Book", regular_str, "__str__ should include class name")
        self.assertIn("333", regular_str, "__str__ should include ISBN")
        
        print("All inheritance tests passed!")

suite = unittest.TestLoader().loadTestsFromTestCase(TestWeek8Inheritance)
runner = unittest.TextTestRunner(verbosity=2)
result = runner.run(suite)

test_loan_periods_and_str (__main__.TestWeek8Inheritance.test_loan_periods_and_str) ... ok

----------------------------------------------------------------------
Ran 1 test in 0.001s

OK
ok

----------------------------------------------------------------------
Ran 1 test in 0.001s

OK


All inheritance tests passed!


## 20) Polymorphic fee scenario (end‑to‑end)

Create a short demo that:
- Builds a `Catalog` and `LoanDesk` (with `Notifier` if you implemented it)
- Adds one book of each subtype
- Checks each out to a `Student`
- Simulates `days_late` values and prints fees using your polymorphic fee API
Show that different subtypes yield different fees without `if/elif` at the call site.

In [39]:
# Your code here

catalog = Catalog()
notifier = Notifier()
desk = NotifyingLoanDesk(catalog, notifier)

books = [
    PrintedBook("PB001", "Printed Python Guide", 3),
    EBook("EB001", "Digital Data Science", 5, 25.0),
    AudioBook("AB001", "Audio Machine Learning", 2, 480),
    Book("RB001", "Regular Statistics Book", 1)
]

for book in books:
    catalog.add_book(book)

print(" Books added to catalog:")
for book in books:
    print(f"  - {book.describe()}")

student = Student("STU001", "student@university.edu")
print(f"\n Member: {student.describe()} (max loans: {student.max_concurrent_loans()})")

# 4. Check each book out to the student
print(f"\n Checking out books:")
loans = []
for book in books:
    try:
        loan = desk.checkout(student, book)
        loans.append((loan, book))
        print(f"   Checked out: {book.title}")
    except LibraryError as e:
        print(f"   Failed: {e}")

print(f"\n Late fee calculations (polymorphic - no if/elif at call site):")
days_late_scenarios = [3, 7, 14]

for days in days_late_scenarios:
    print(f"\n  Days late: {days}")
    for loan, book in loans:
    
        fee = compute_fee(book, days)
        print(f"    {book.__class__.__name__:12} '{book.title}': ${fee:.2f}")


print(f"\n📊 Fee comparison for 5 days late:")
test_books = [
    PrintedBook("TEST1", "Test Printed", 1),    
    EBook("TEST2", "Test EBook", 1, 10.0),      
    AudioBook("TEST3", "Test Audio", 1, 120),   
    Book("TEST4", "Test Regular", 1)            
]

for book in test_books:
    fee = compute_fee(book, 5)  
    print(f"  {book.__class__.__name__:12}: ${fee:.2f} (${fee/5:.2f}/day)")

print(f"\n🎯 Demo complete! Notice how different book types produce different fees")
print(f"   without any if/elif statements in the fee calculation code.")
print(f"   This is the power of polymorphism! 🚀")

 Books added to catalog:
  - PrintedBook<PB001>: Printed Python Guide (copies=3)
  - EBook<EB001>: Digital Data Science (copies=5, size=25.0MB)
  - Book<AB001>: Audio Machine Learning (copies=2) (duration=480min)
  - Book<RB001>: Regular Statistics Book (copies=1)

 Member: Member<STU001> (max loans: 5)

 Checking out books:
To student@university.edu: You checked out 'Printed Python Guide' due on 2025-11-30
   Checked out: Printed Python Guide
To student@university.edu: You checked out 'Digital Data Science' due on 2025-11-23
   Checked out: Digital Data Science
To student@university.edu: You checked out 'Audio Machine Learning' due on 2025-11-23
   Checked out: Audio Machine Learning
To student@university.edu: You checked out 'Regular Statistics Book' due on 2025-11-23
   Checked out: Regular Statistics Book

 Late fee calculations (polymorphic - no if/elif at call site):

  Days late: 3
    PrintedBook  'Printed Python Guide': $0.75
    EBook        'Digital Data Science': $0.30
    

## Python skills you'll need (Weeks 1–8)

- **Core syntax & data types:** variables, strings, numbers, booleans
- **Collections:** lists, dicts (basic use), simple list/dict comprehensions
- **Control flow:** `if/elif/else`, `for`, `while`
- **Functions & modules:** defining functions, parameters, returns, imports
- **File I/O & JSON (basic):** open/read/write, simple JSON usage
- **Classes & objects (Weeks 4–6):** defining classes, attributes, methods, `__init__`, `__str__`
- **Encapsulation basics:** simple validation; naming conventions for "private" attributes
- **Methods:** instance/class/static methods (as introduced up to Week 6)
- **Error handling & testing (Week 7):** `try/except/else/finally`, custom exceptions, basic `unittest`
- **Week 8 focus:** **single inheritance**, **method overriding**, **`super()`**, **polymorphism via common methods**, and **composition vs. inheritance** decisions
- **Standard library familiarity:** `datetime`, `timedelta`, built‑in exceptions
